<a href="https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task type: ranking / scoring. The decision is which pages should be reviewed first for a possible refresh or other content action. A ranked score fits this decision better than a yes/no classification because editorial time is limited: the useful output is an ordered review queue.

I will start with a transparent fixed rule. ML is only justified if combining multiple observable signals produces a better ranking on future held-out data without adding unnecessary complexity.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from pathlib import Path
import pandas as pd

REPO_DIR = Path("/content/flyrank-internship-ml")
REPO_URL = "https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml"

# Clone the repository in Colab if it is not already present
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

# Move into the repository
os.chdir(REPO_DIR)

print("Repository ready.")
print("Working directory:", Path.cwd())

# Load the dataset once for the whole notebook
DATA_PATH = REPO_DIR / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded: {len(df):,} rows × {len(df.columns):,} columns")

Repository ready.
Working directory: /content/flyrank-internship-ml
Dataset loaded: 30,000 rows × 44 columns


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Primary target for the full warehouse: an observed future decline measured in a later 30-day window. After the feature window ends, the target can be defined from the observed future data—for example, future 30-day impressions falling more than 20% versus the preceding 30-day comparison window.

Starter-data proxy: trend_direction == "down" is an observed current/trailing-window decline indicator. It is useful for checking the starter data, but it is not a valid future target for a forward-looking model because it is created from the current window.

The important distinction is that the model should learn from an outcome that occurs after the decision point, rather than learn to reproduce an existing rule.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).

current_decline_proxy = df["trend_direction"].eq("down")

print(
    f"Current decline proxy: "
    f"{current_decline_proxy.sum():,} / {len(df):,} pages "
    f"({current_decline_proxy.mean():.1%})"
)

print("Future 30-day target available in starter snapshot: no")
print(
    "Leakage check: trend_direction and trend_pct "
    "will be excluded from model features."
)


Current decline proxy: 16,262 / 30,000 pages (54.2%)
Future 30-day target available in starter snapshot: no
Leakage check: trend_direction and trend_pct will be excluded from model features.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The primary success metric will be Precision@20. It is the proportion of the top 20 pages in the ranked review queue that truly match the future target. This matches the decision because an editor may only have capacity to inspect a small number of pages first.

A practical success criterion is that the ML ranking should improve Precision@20 over the transparent fixed-rule baseline on a future, time-aware holdout. I will not claim a numeric improvement from this starter snapshot because its future target is not available.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

review_capacity = 20
print(f"Primary evaluation metric: Precision@{review_capacity}")
print("Success criterion: beat the fixed-rule baseline on the same held-out future period.")


Primary evaluation metric: Precision@20
Success criterion: beat the fixed-rule baseline on the same held-out future period.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one pseudonymized content item/page per row. This matches the action because an SEO or content editor reviews individual pages. content_id is an identifier for grouping and integrity checks only; it will not be used as a predictive feature.

The starter slice is already at this page-level grain, so it is appropriate for checking the structure before moving to the warehouse's time-based data for future-target construction.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

lane_columns = [
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
]

lane_df = df[lane_columns].copy()

print("Shape:", lane_df.shape)
print("One row = one content item/page")
print("Unique page IDs in source:", df["content_id"].nunique())
print("Average position coded as 0 (no position data):", (df["avg_position"] == 0).sum())
print("\nPreview of the lane slice:")
print(lane_df.head(5).to_string(index=False))


Shape: (30000, 9)
One row = one content item/page
Unique page IDs in source: 30000
Average position coded as 0 (no position data): 1205

Preview of the lane slice:
   content_type  impressions_90d  clicks_90d  sessions_90d  content_age_days  days_since_last_update  ctr  avg_position  engagement_rate
keyword article             3803          29            17               187                      20 0.76          10.6             5.88
keyword article            15320           7             9               445                      25 0.05          20.3             0.00
keyword article            12581          11            11               141                      20 0.09          36.5             0.00
keyword article            11751          58            78               463                      22 0.49           6.2             1.28
keyword article            19140          24           145               263                      14 0.13          44.0             0.00


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The case for ML is not that a simple rule is useless. A fixed rule can capture obvious candidates, such as pages with a meaningful decline and enough search visibility. The harder cases involve several signals that can disagree: visibility, CTR, position, content age, freshness, keyword demand, content type, and intent can all change the priority of a page. Their interactions are difficult to express cleanly with many hand-written thresholds.

Therefore, ML is worth testing because it may learn useful combinations of these observable signals. It only earns its place if it improves Precision@20 on a future holdout enough to justify the extra complexity. If it does not, the transparent rule remains the better decision-support method.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

signal_summary = pd.DataFrame({
    "signal": [
        "content_type", "impressions_90d", "avg_position",
        "ctr", "content_age_days", "days_since_last_update",
        "engagement_rate"
    ],
    "distinct_or_missing": [
        df["content_type"].nunique(),
        df["impressions_90d"].nunique(),
        df["avg_position"].nunique(),
        df["ctr"].nunique(),
        df["content_age_days"].nunique(),
        df["days_since_last_update"].nunique(),
        df["engagement_rate"].nunique(),
    ]
})

print("Observable signals have multiple values and can contribute different pieces of evidence:")
print(signal_summary.to_string(index=False))
print("\nThis supports testing a multi-signal model, not claiming in advance that ML will win.")


Observable signals have multiple values and can contribute different pieces of evidence:
                signal  distinct_or_missing
          content_type                    3
       impressions_90d                 9438
          avg_position                  869
                   ctr                  401
      content_age_days                  225
days_since_last_update                   57
       engagement_rate                  915

This supports testing a multi-signal model, not claiming in advance that ML will win.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.